In [ ]:
%pip install pandas openpyxl


In [ ]:
import pandas as pd

df = pd.read_excel("../data/raw/dataset.xlsx")  # or read_csv
df.shape

In [ ]:
df.columns.tolist()


In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
binary_cols = ['HLA-B27', 'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm']
for col in binary_cols:
    print(col, df[col].unique())

In [ ]:
missing = df.isna().sum().to_frame('n_missing')
missing['pct_missing'] = (missing['n_missing'] / len(df) * 100).round(1)
missing.sort_values('pct_missing', ascending=False)

In [ ]:
%pip install scipy

In [ ]:
# Test whether missingness in each lab marker is random or disease-dependent (MCAR vs MAR).
# H0: missingness is independent of Disease. If p < 0.05, we reject H0 — missingness is
# NOT random, meaning doctors likely ordered/skipped this test based on suspected diagnosis.
# In that case MICE would impute a value doctors deliberately chose not to measure, so we
# flag it with a _was_missing indicator instead of blindly imputing.
import pandas as pd
from scipy.stats import chi2_contingency

cols_to_check = ['ESR', 'CRP', 'RF', 'Anti-CCP', 'HLA-B27', 'ANA',
                  'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm', 'C3', 'C4']
results = []
for col in cols_to_check:
    is_missing = df[col].isna()
    contingency = pd.crosstab(is_missing, df['Disease'])
    chi2, p, dof, _ = chi2_contingency(contingency)
    results.append({'feature': col, 'chi2': round(chi2, 2), 'p_value': p})
result_df = pd.DataFrame(results).sort_values('p_value')
result_df

In [ ]:
pd.crosstab(df['RF'].isna(), df['Disease'], normalize='columns').round(3) * 100

In [ ]:
# outlier check
# there are lots of outliers in this data because disease patients are expected to have out of range test stats
# we cant remove these
ranges = {
    'CRP': (0.1, 3.0),
    'RF': (0.1, 3.0),
    'Anti-CCP': (0.0, 20.0),
}

for col, (lo, hi) in ranges.items():
    out_of_range = ~df[col].between(lo, hi) & df[col].notna()
    print(f"{col}: {out_of_range.sum()} values out of documented range ({lo}-{hi}), out of {df[col].notna().sum()} non-null")

In [ ]:
esr_lo_m, esr_hi_m = 0, 15
esr_lo_f, esr_hi_f = 0, 20

male_mask = df['Gender'] == 'Male'
out_of_range = (
    (male_mask & ~df['ESR'].between(esr_lo_m, esr_hi_m)) |
    (~male_mask & ~df['ESR'].between(esr_lo_f, esr_hi_f))
) & df['ESR'].notna()

print(f"ESR: {out_of_range.sum()} out of range, out of {df['ESR'].notna().sum()} non-null")

In [ ]:
# remove any valies that are physiologically impossible as they are errors
continuous_cols = ['ESR', 'CRP', 'RF', 'Anti-CCP', 'C3', 'C4']

for col in continuous_cols:
    print(f"{col}: min={df[col].min()}, max={df[col].max()}")

In [ ]:
%pip install matplotlib seaborn


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

continuous_cols = ['ESR', 'CRP', 'RF', 'Anti-CCP', 'C3', 'C4']

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
axes = axes.flatten()

for i, col in enumerate(continuous_cols):
    sns.boxplot(data=df, x='Disease', y=col, ax=axes[i])
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

**ESR and CRP show an odd, very clean split: {RA, Reactive Arthritis, AS, PsA} all sit high (20–40ish) while {Sjögren's, SLE, Normal} all sit low (0–10), with almost zero overlap between the two groups.** This is the part that doesn't fully match clinical expectation — Sjögren's and SLE are inflammatory autoimmune diseases too, and grouping them with *Normal* on inflammation markers is unusual. It's such a clean, near-binary split that it's worth ruling out a **data collection artifact** rather than assuming it's pure biology.

Recall the dataset was pooled from 4 different sources (Saint Raphael Hospital, Al Hawraa Clinical, Al-Sibtain Medical, Taqadum), collected at different times. If, say, all the Sjögren's/SLE cases happened to come predominantly from one lab that used a different assay/units, or recorded milder-stage patients, that could produce exactly this kind of clean group split that isn't really about the disease at all.

We don't have a `source_lab` column in your dataframe, so we can't test this directly — but it's worth flagging honestly in your paper's limitations regardless: *"ESR/CRP showed near-complete separation into two disease clusters not fully aligned with known inflammatory biology, possibly reflecting cohort/collection differences not captured in the available features."* That's a legitimate, honest observation — better to name it than let a reviewer spot it first.

In [ ]:
binary_cols = ['HLA-B27', 'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm']

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
axes = axes.flatten()

for i, col in enumerate(binary_cols):
    positivity = df.groupby('Disease')[col].apply(lambda x: (x == 'Positive').mean() * 100)
    positivity.sort_values().plot(kind='barh', ax=axes[i])
    axes[i].set_title(f'{col} positivity rate (%)')
    axes[i].set_xlabel('% Positive')

plt.tight_layout()
plt.show()

HLA-B27 → clearly spikes for Reactive Arthritis (~83%) and AS (~82%), both spondyloarthropathies, while every other class sits in a tight 30-45% band. This is exactly the textbook pattern — and interestingly, it also hints at why AS/RA/Reactive/PsA get confused with each other in the benchmark paper: HLA-B27 is shared strongly by two of your seronegative spondyloarthropathy diseases, but doesn't cleanly separate them from each other, only from the rest.
ANA → correctly spikes for Sjögren's (~68%) and SLE (~66%), the two classic ANA-positive diseases, distinctly above everything else. Clean.
Anti-Ro → sharply spikes for Sjögren's (~83%), exactly as expected — Anti-Ro is a defining Sjögren's marker.
Anti-La → spikes for Sjögren's (~72%), again textbook — Anti-La is Sjögren's-specific, matching the literature (higher specificity than Anti-Ro, and indeed here it looks even more concentrated in Sjögren's alone, with SLE a distant second).
Anti-dsDNA → spikes hard for SLE (~62%), clean and specific, exactly matching the literature (Anti-dsDNA is one of the more SLE-specific serology tests).
Anti-Sm → spikes very strongly for SLE (~55%), again a known SLE-specific marker, cleanly separated from everything else.
This is a really useful diagnostic pattern to notice now, before you even train anything: Sjögren's and SLE are going to be relatively easy (multiple strong, near-unique binary markers each: ANA+Anti-Ro+Anti-La for Sjögren's; ANA+Anti-dsDNA+Anti-Sm+low-C3/C4 for SLE). But AS, Reactive Arthritis, and PsA share the same strong signal (HLA-B27) with nothing else clearly separating them from one another — so your model is going to lean almost entirely on the weaker, less-clean continuous features (ESR/CRP/RF) to tell those three apart, which is exactly where the benchmark paper's AS-recall problem came from. This is good context to have written down now, so when you see the same confusion later in your own confusion matrix, you already know why — it's not a modeling failure, it's a feature-availability ceiling.

In [ ]:
# PCA and plotting
continuous_cols = ['Age', 'ESR', 'CRP', 'RF', 'Anti-CCP', 'C3', 'C4']
corr = df[continuous_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation matrix — continuous features')
plt.show()

What's expected and good
RF ↔ Anti-CCP: 0.40 — moderate positive correlation, matches the known clinical co-occurrence (both are RA-associated autoantibodies, commonly elevated together). Not surprising, not a problem.
C3 ↔ C4: 0.35 — moderate positive correlation, also expected (both complement proteins, tend to move together, though we saw them both crash specifically in SLE from the boxplots).
Age is essentially uncorrelated with everything (all ~0.00-0.02) — matches the benchmark paper finding that Age had minimal SHAP importance. Consistent, not concerning.
C3/C4 vs ESR/CRP (~0.22-0.24) — mild, plausible (general inflammation loosely relates to complement activity), nothing alarming.

In [ ]:
pip install scikit-learn

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# quick-and-dirty for visualization only: drop rows with any missing continuous value
# (this is NOT how we'll handle missingness in the real pipeline — just for this one plot)
pca_df = df[continuous_cols[1:] + ['Disease']].dropna()  # excluding Age for now, or include it if you want

X = pca_df[continuous_cols[1:]]
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 8))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=pca_df['Disease'], alpha=0.5, palette='tab10')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('PCA of continuous features, colored by diagnosis')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

Left-to-right (PC1) is basically an inflammation axis. Looking at where each color sits: SLE and Sjögren's (purple, red) cluster on the left, while RA and AS (blue, green) cluster on the right. This lines up exactly with what we saw in the ESR/CRP boxplots earlier — remember that odd clean split where {RA, Reactive, AS, PsA} had high ESR/CRP and {Sjögren's, SLE, Normal} had low ESR/CRP? PC1 is essentially that same signal, now visualized. So this isn't a new finding — it's the same inflammation-marker split, just confirmed from a different angle.

Top-to-bottom (PC2) is roughly a second axis, less obvious what it represents alone — likely a mix of RF/Anti-CCP and C3/C4 pulling in different directions, but PCA doesn't label this for us in plain terms, it's just "the second most useful direction to spread points apart."

SLE (purple) is the most visually distinct cluster — sits in its own zone, upper-left, with relatively little overlap into other colors. This matches everything we've seen so far: strong ANA/Anti-dsDNA/Anti-Sm positivity, distinctly low C3/C4, and now distinct on continuous features too. This is almost certainly going to be your easiest class, same as in the benchmark paper (97.9% recall there).
Normal (brown) sits centrally, low-inflammation, but heavily overlapping with Sjögren's (red). That's a real and slightly concerning pattern — it suggests some Normal (control) patients and some Sjögren's patients look nearly identical on just these 6 continuous markers. Makes sense biologically actually — Sjögren's often presents mildly, with modest inflammation, so a mild/early Sjögren's case can genuinely look "normal" on bloodwork alone. This is exactly the kind of confusion your binary markers (ANA, Anti-Ro, Anti-La) will need to resolve, since PC1/PC2 alone won't cut it for this pair.
RA (blue), AS (green), and PsA (pink) are heavily intermixed on the right side — no clean boundaries between them at all in this view. This is the cluster to pay attention to, because it's a three-way tangle, not just two-way. This directly foreshadows the confusion matrix problem from the benchmark paper (AS→RA was their single biggest error category, 28.6% of all mistakes) — and now you can see visually why: on these continuous features alone, these three diseases are nearly indistinguishable. Your model is going to be leaning heavily on HLA-B27 (shared by AS/Reactive, but not RA/PsA as strongly) and RF/Anti-CCP (elevated broadly, not cleanly RA-specific per our earlier boxplots) to pull these apart — and we already saw those markers aren't as sharp here as textbook rheumatology would suggest.
Reactive Arthritis (orange) is barely visible — it's the smallest class (516 patients) and its dots seem to scatter thinly across the RA/AS/PsA tangle rather than forming any cluster of its own. Given how few points there are, combined with sitting inside the messiest region of the plot, this is likely going to be your hardest class to classify well — worth expecting low recall here specifically, and not being surprised by it later.